# Tokenizer
We started to experiment with `Romance of Three Kindoms`

In [1]:
import requests
import pandas as pd
from pathlib import Path
import json
import regex
import time

In [2]:
pat = regex.compile(
    r"[\p{P}\p{S}]|'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+[\p{L}&&\n^\p{Han}]++|[\p{Han}]{1,2}+|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s")

In [3]:
def get_locations(corpus):

    DATA = Path("../data")
    DATA.mkdir(exist_ok=True)
    TK_DATA = DATA / "tokenizer"
    TK_DATA.mkdir(exist_ok=True)
    CORPUS_DATA = TK_DATA / corpus
    CORPUS_DATA.mkdir(exist_ok=True)
    return CORPUS_DATA

In [4]:
CORPUS = "rtk"
CORPUS_DATA = get_locations(CORPUS)

In [5]:
if CORPUS =="xyj":
    url = "https://raw.githubusercontent.com/tennessine/corpus/refs/heads/master/%E8%A5%BF%E6%B8%B8%E8%AE%B0.txt"

elif CORPUS == "wm":
    url = "https://github.com/tennessine/corpus/raw/refs/heads/master/%E6%B0%B4%E6%B5%92%E4%BC%A0.txt"

elif CORPUS == "rtk":
    url = "https://github.com/tennessine/corpus/raw/refs/heads/master/%E4%B8%89%E5%9B%BD%E6%BC%94%E4%B9%89.txt"

In [6]:
def get_text(url):
    if (CORPUS_DATA / "corpus.txt").exists() is False:
        text = requests.get(url).text
        with open(CORPUS_DATA / "corpus.txt", "w") as f:
            f.write(text)
    else:
        text = (CORPUS_DATA / "corpus.txt").read_text()

    return text

In [7]:
text = get_text(url)

In [8]:
text[:100]

'《三国演义》罗贯中\r\n\r\n第一回 宴桃园豪杰三结义 斩黄巾英雄首立功\r\n\r\n    \t\t\r\n\r\n    滚滚长江东逝水，浪花淘尽英雄。是非成败转头空。\r\n\r\n    青山依旧在，几度夕阳红。白发渔樵江'

In [9]:
len(text)

608315

In [10]:
utf8 = list(text.encode())

In [11]:
len(utf8)

1795281

In [12]:
freq = dict()

for c1, c2 in zip(utf8,utf8[1:]):
    freq[(c1, c2)] = freq.get((c1, c2),0) + 1

In [13]:
most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])
most_freq[:5]

[((239, 188), 61651),
 ((188, 140), 43301),
 ((228, 184), 31246),
 ((227, 128), 27554),
 ((128, 130), 24346)]

In [14]:
# code_list = list(utf8)

code_collection = list(list(sub_text.encode("utf8")) for sub_text in pat.findall(text))

codes_start = 0
utf_len = len(list(text.encode("utf8")))
for l in code_collection:

    codes_start = max(max(l), codes_start)

vocab = dict()


def merge(ids, temp_vocab):
    if len(ids) < 2:
        return ids
    new_ids = []
    i = 0
    while i < len(ids):
        if i == len(ids) - 1:
            new_ids.append(ids[i])
            break
        token = temp_vocab.get((ids[i], ids[i + 1]))
        if token is not None:
            new_ids.append(token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

logs = []

last_ts = time.time()
while True:
    freq = dict()

    for code_list in code_collection:

        if len(code_list) < 2:
            continue

        for c1, c2 in zip(code_list, code_list[1:]):
            freq[(c1, c2)] = freq.get((c1, c2), 0) + 1

    most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])

    temp_vocab = dict()
    top_freq_ids1 = set()
    top_freq_ids2 = set()

    for (c1, c2), ct in most_freq:
        if c1 in top_freq_ids2 or c2 in top_freq_ids1 or ct < 5:
            break
        top_freq_ids1.add(c1)
        top_freq_ids2.add(c2)

        codes_start += 1

        temp_vocab[(c1, c2)] = codes_start

        vocab[codes_start] = (c1, c2)

    # print(temp_vocab)

    if len(temp_vocab) == 0:
        print(f"\nFinished Vocab:{len(vocab)} /{ct} / {ct/utf_len:.5f}")
        break

    # print(f"Try to merge {len(temp_vocab)}")
    new_code = codes_start

    if len(vocab) % 100 == 0:
        print(f"V:{len(vocab)}x", end="\t")

    ct = 0

    for i in range(len(code_collection)):
        code_collection[i] = merge(code_collection[i], temp_vocab)
        ct += len(code_collection[i])

    new_ts = time.time()

    logs.append({
        "vocab_len": len(vocab),
        "code_list_len": ct,
        "compression": ct / utf_len,
        "time": new_ts - last_ts,
    })

    last_ts = new_ts
    

V:2800x	V:4800x	
Finished Vocab:13345 /4 / 0.00000


In [15]:
df = pd.DataFrame(logs)
df

,vocab_len,code_list_len,compression,time
0,1,1733630,0.965659,0.524506
1,4,1631529,0.908788,0.527753
2,14,1485341,0.827359,0.497530
3,35,1342545,0.747819,0.445650
4,48,1285571,0.716083,0.420467
...,...,...,...,...
613,13313,416699,0.232108,0.232595
614,13314,416694,0.232105,0.179510
615,13338,416574,0.232038,0.245277
616,13339,416569,0.232036,0.190557


In [16]:
with open(CORPUS_DATA / "vocab.json", "w") as f:
    f.write(json.dumps(vocab, indent=2), )

with open(CORPUS_DATA / "freq.json", "w") as f:
    f.write(json.dumps(most_freq, indent=2))

In [17]:
df = pd.DataFrame(logs)
df.to_csv(str(CORPUS_DATA / "logs.csv"), index=False)

## Retro Analysis

In [18]:
import json
import pandas as pd

In [19]:
df = pd.read_csv(str(CORPUS_DATA / "logs.csv"))

In [20]:
with open(CORPUS_DATA/"vocab.json") as f:
    vocab = dict((int(k), v) for k, v in json.loads(f.read()).items())

with open(CORPUS_DATA/"freq.json") as f:
    most_freq = json.loads(f.read())

In [21]:
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_df(df):
    cols = ["vocab_len", "code_list_len", "compression", "time"]
    titles = ["Vocab Size", "Code List Length", "Compression Ratio", "Time per Step (s)"]

    fig = make_subplots(rows=2, cols=2, subplot_titles=titles)

    for i, (col, title) in enumerate(zip(cols, titles)):
        row, col_idx = divmod(i, 2)
        fig.add_trace(
            go.Scatter(x=df.index, y=df[col], mode="lines", name=title),
            row=row + 1, col=col_idx + 1,
        )

    fig.update_layout(height=600, title_text="BPE Tokenizer Training", showlegend=False)
    fig.show()


In [22]:
plot_df(df)

In [23]:
pair_to_code = dict((tuple(v), int(k)) for k, v in vocab.items())

In [24]:
from loguru import logger

In [25]:
def encode(text: str, pair_to_code):
    code_list = list(text.encode("utf8"))
    if len(code_list) < 2:
        return code_list
    while len(code_list) > 1:
        temp_vocab = dict()
        for c1, c2 in zip(code_list, code_list[1:]):
            token = pair_to_code.get((c1, c2))
            if token is None:
                continue
            else:
                temp_vocab[(c1, c2)] = token
        if len(temp_vocab) == 0:
            return code_list


        pair = min(temp_vocab, key=lambda x: temp_vocab[x])
        token = temp_vocab[pair]

        new_code_list = []

        i = 0
        while i < len(code_list):
            if i < len(code_list) - 1 and code_list[i] == pair[0] and code_list[i + 1] == pair[1]:
                logger.info(f"[REPLACE]{code_list[i]}, {code_list[i + 1]} -> {token}")
                new_code_list.append(token)
                i += 2
            else:
                new_code_list.append(code_list[i])
                i += 1
        
        code_list = new_code_list
            
    return code_list

In [30]:
list("华容道".encode("utf8"))

[229, 141, 142, 229, 174, 185, 233, 129, 147]

In [31]:
token_ids = encode("华容道", pair_to_code=pair_to_code)
print(token_ids)

2026-06-25 21:17:26.510 | INFO     | __main__:encode:25 - [REPLACE]229, 141 -> 264
2026-06-25 21:17:26.511 | INFO     | __main__:encode:25 - [REPLACE]229, 174 -> 266
2026-06-25 21:17:26.511 | INFO     | __main__:encode:25 - [REPLACE]233, 129 -> 312
2026-06-25 21:17:26.512 | INFO     | __main__:encode:25 - [REPLACE]312, 147 -> 627
2026-06-25 21:17:26.513 | INFO     | __main__:encode:25 - [REPLACE]266, 185 -> 1000
2026-06-25 21:17:26.514 | INFO     | __main__:encode:25 - [REPLACE]264, 142 -> 1204
2026-06-25 21:17:26.514 | INFO     | __main__:encode:25 - [REPLACE]1204, 1000 -> 6401
2026-06-25 21:17:26.515 | INFO     | __main__:encode:25 - [REPLACE]6401, 627 -> 9957


[9957]


In [33]:
token_ids = encode("玄德", pair_to_code=pair_to_code)
print(token_ids)

2026-06-25 21:17:51.618 | INFO     | __main__:encode:25 - [REPLACE]229, 190 -> 265
2026-06-25 21:17:51.618 | INFO     | __main__:encode:25 - [REPLACE]231, 142 -> 295
2026-06-25 21:17:51.619 | INFO     | __main__:encode:25 - [REPLACE]265, 183 -> 342
2026-06-25 21:17:51.619 | INFO     | __main__:encode:25 - [REPLACE]295, 132 -> 362
2026-06-25 21:17:51.620 | INFO     | __main__:encode:25 - [REPLACE]362, 342 -> 368


[368]


In [34]:
def decode(tokens, vocab):
    result = list(tokens)
    
    while True:
        i = 0
        new_result = []
        temp_vocab = dict((i, vocab[i]) for i in result if i in vocab)
        if len(temp_vocab) == 0:
            break
        token = max(temp_vocab, key=lambda x: x)
        c1, c2 = temp_vocab[token]
        for t in result:
            if t == token:
                new_result.append(c1)
                new_result.append(c2)
            else:
                new_result.append(t)
        result = new_result
        
    return bytes(result).decode(errors="replace")


def by_token_decode(tokens, vocab):
    results = []
    for token in tokens:
        by_token_str = decode([token], vocab)
        results.append(by_token_str)
    return results
    

In [35]:
for i in range(1000, 2000):
    print(
        decode([i], vocab=vocab),
        end="\n" if i % 30 == 0 else ", ")

容, �下, 诗, 奈, 之事, 七, 宁, 举, �, 叔, 保, 汉中, �, 止, 世, 贤, 夺, 都督, 孤, �, 衣
甲, 神, 虚, 马超, 付, 陛下, 断, 蒙, 土, 幸, 居, 谁, 达, 星, 指, 承, 对, 征, �, �, 侍, 召, 威, 史, 天子, 只见, �, 程, 恩, 渡
后主, 食, 元, 散, 恨, 全, 震, 母, 翼, 骂, 一军, 顾, 称, 落, 陵, 慌, 陆, 卿, 目, 色, 聚, 新, 上马, 曾, 勇, 谏, 岱, 晃, 速, 氏
驾, 黄忠, 遇, 有一, 特, 建, 约, 府, 昔, �, 危, 击, 璋, 六, 刺, 朕, 张郃, 背后, 丕, 亡, 石, �, 临, 拨, 大叫, 喝, 伐, 料, 城中, 不得
忧, 女, 渊, 干, 逊, 而来, 之言, 青, 术, 弓, 先主, 逃, 赏, 叱, 队, 和, 细, 果, 寻, �, 英, 蔡, 精, 赐, 桥, �, 使人, 盖, 诈, 离
�, 否, 蛮, 太守, 此人, 甘, 绝, 忽报, 祖, 光, 善, 顺, 淮, 襄, 倒, 一面, 牛, 步, 严, 大军, 结, 推, 后人, 袭, 班, 登, 堂, 何不, 先生, 有诗
物, 木, 依, 俱, 何故, 宴, 加, 不如, 角, 九, 负, 盛, 松, 息, 执, 令人, 然后, 匹, 泰, 惧, 苞, 罢, 仇, 突, 邓艾, 拒, 理, 旧, �, 劳
情, 先锋, 孟获, 华, 慈, 的, 之计, 诸葛亮, 典, 惇, 睿, 禁, 位, 岸, 赴, �, 奏曰, 颜, 招, 二十, 下马, 劫, 劝, 叹曰, 一声, 褚, 造, 挺, 出城, 敬
汝等, 原来, 驱, 妻, 洪, 县, 儿, �, 厚, 赶来, 群, 异, 之人, 室, 霸, 修, 宗, 后人有诗, 忙, 伤, 江东, 灵, 入城, �, 持, 曹仁, 大将, 亭, 利, 奈何
满, 二将, 张辽, 瑾, 护, 军中, 业, 拔, 尉, 语, 在此, 莫, �, 怀, 灭, 致, �, 领兵, 正是, 之兵, 仪, �山, 成都, 讨, 大事, 初, 忽然, 化, 变, 徐晃
之后, �, 喊声, 野, 晚, 傕, 寨中, 空, 御, 殿, 以为, 大败, 不出, 徐州, 险, 不见, 郎, 通, 倘, 汜

In [38]:
for i in range(12345, 13345):
    try:
        print(
            decode([i], vocab=vocab),
            end="\n" if i % 30 == 0 else "| ")
    except ValueError:
        print(i)

起兵十万| 以讨| 下将| 自送| 此一| 却都| 到城下| 招呼| 霸王| 捉了| 欲起兵| 屯小沛| 听其言| 令人送| 礼贤| 下士
孙郎| 复何| �县| 往江东| 扬州刺史| 寇将军| 带领| 昆| 许允| 字正| 纵有| 不喜| 引兵到| 滩| 收得| 遂进兵| 神亭| 笮| 笮融| 上岭| 祝曰| 探看| 半点| 粉碎| 背上| 风雨| 把枪| 某自| 挺枪直取| 连夜投
豫章| 部兵| 瑜令| 新破| 日中| 听从| 金石| 弓弦响处| 令兄| 来接| 引兵救| 接住| 掩之| 不战而| 烟火| 吾随后| 数枪| 医者| 待为上宾| 以药| 各处隘口| 杨大将| 十六回| 擒刘备| 可得也| 回告| 欲报| 不救| 吾想| 下营寨
西南上| 无信| 特请| 大耳| 居左| 居右| 你两家| 惭愧| 玄德拜谢| 辕门射| 不可造次| 为媒| 则吾| 止有一| 公台| 识破| 甚当| 之父| 养老| 珪曰| 女儿| 招军买马| 正话间| 抵赖| 恨者| 玄德令| 以上宾| 劝我| 俊杰| 豫州牧
得罪| 识吾| 乃天| 夫人去| 取乐| 其军| 步卒| 奋力向前| 门而入| 三箭| 曹昂| 本部军| 即至| 之名将| 非公| 益州刘璋| 掷剑| 十七回| 其二| 纣| 陈纪| 为太尉| 沂都| 以求| 擒下| 亲信| 来此何干| 分兵五路| 一队军马| 伤其
逆不| 加兵| 令曹仁| 北面| 尽数| 接济| 不敷| 汝必| 吝| 一刀斩| 破城| 吾令汝| 排銮驾| 甚严| 相传| 布袋| 多半| 锹| 计点| 被伤| 湖口| 阵亡| 如其不然| 为备| 其不备| 遣使赍诏| 何以知| 不足惧| �| 繁
此谋| 不能答| 来使曰| 备当| 钧命| 取汝| 军至| 低头不语| 曹兵大败| 亦至| 未知玄德| 忽见一| 提大军| 脱身之计| 退步| 劝主公| 箭上| 某已| 在敌楼上| 却只| 张辽也| 不能取胜| 亲统大军| 商议起兵| 安心| 吾方| 操谓| 不失| 归府| 不如坚守
无及矣| 之福| 在意| 眭| 北有| 军无| 禀曰| 哀告| 宪曰| 吾当先| 抗拒| 公自| 大声| 栋梁| 公为| 保之| 之耳| 已降| 厚赏| 操封| 大犒三军| 封爵| 操唤| 出朝| 当即| 文武百官| 不敢动| 心腹之人| 难行| 岂不可
因指| 张良| 

Turn down the logging level a little bit

In [39]:
import sys
logger.remove()
logger.add(sys.stderr, level="WARNING")

1

In [40]:
token_ids = encode("中山靖王之后", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['中山靖', '王之后']

In [ ]:
token_ids = encode("", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['青龙', '偃月', '刀']

In [42]:
token_ids = encode("喝令左右推出斩之", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['喝令左右推出斩之']

In [43]:

token_ids = encode("hello", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['h', 'e', 'l', 'l', 'o']

In [44]:
print(decode(encode(text[30000:31000], pair_to_code), vocab))

我搜看。”坚怒曰：“汝有何力，敢小觑我！”方欲交兵，刘表便退。坚纵马赶去，两山后伏兵齐起，背后蔡瑁、蒯越赶来，将孙坚困在垓心。正是：玉玺得来无用处，反因此宝动刀兵。毕竟孙坚怎地脱身，且听下文分解。



第七回 袁绍磐河战公孙 孙坚跨江击刘表

    		

    却说孙坚被刘表围住，亏得程普、黄盖、韩当三将死救得脱，折兵大半，夺路引兵回江东。自此孙坚与刘表结怨。

    且说袁绍屯兵河内，缺少粮草。冀州牧韩馥，遣人送粮以资军用。谋士逢纪说绍曰：“大丈夫纵横天下，何待人送粮为食！冀州乃钱粮广盛之地，将军何不取之？”绍曰：“未有良策。”纪曰：“可暗使人驰书与公孙瓒，令进兵取冀州，约以夹攻，瓒必兴兵。韩馥无谋之辈，必请将军领州事；就中取事，唾手可得。”绍大喜，即发书到瓒处。瓒得书，见说共攻冀州，平分其地，大喜，即日兴兵。

    绍却使人密报韩馥。馥慌聚荀谌、辛评二谋士商议。谌曰：“公孙瓒将燕、代之众，长驱而来，其锋不可当。兼有刘备、关、张助之，难以抵敌。今袁本初智勇过人，手下名将极广，将军可请彼同治州事，彼必厚待将军，无患公孙瓒矣。”韩馥即差别驾关纯去请袁绍。长史耿武谏曰：“袁绍孤客穷军，仰我鼻息，譬如婴儿在股掌之上，绝其乳哺，立可饿死。奈何欲以州事委之？此引虎入羊群也。”馥曰：“吾乃袁氏之故吏，才能又不如本初。古者择贤者而让之，诸君何嫉妒耶？”耿武叹曰：“冀州休矣！”于是弃职而去者三十余人。独耿武与关纯伏于城外，以待袁绍。

    数日后，绍引兵至。耿武、关纯拔刀而出，欲刺杀绍。绍将颜良立斩耿武，文丑砍死关纯。绍入冀州，以馥为奋威将军，以田丰、沮授、许攸、逢纪分掌州事，尽夺韩馥之权。馥懊悔无及，遂弃下家小，匹马往投陈留太守张邈去了。

    却说公孙瓒知袁绍已据冀州，遣弟公孙越来见绍，欲分其地。绍曰：“可请汝兄自来，吾有商议。”越辞归。行不到五十里，道旁闪出一彪军马，口称：“我乃董丞相家将也！”乱箭射死公孙越。从人逃回见公孙瓒，报越已死。瓒大怒曰：“袁绍诱我起兵攻韩馥，他却就里取事；今又诈董卓兵射死吾弟，此冤如何不报！”尽起本部兵，杀奔冀州来。

    绍知瓒兵至，亦领军出。二军会于磐河之上：绍军于磐河桥东，瓒军于桥西。瓒立马桥上，大呼曰：“背义之徒，何敢卖我！”绍亦策马至桥边，指瓒曰：“韩馥无才，愿让冀州于吾，与尔


In [45]:

token_ids = encode(text[30000:31000], pair_to_code=pair_to_code)
print("|".join(by_token_decode(token_ids, vocab=vocab)))

我|搜|看|。|”|坚|怒曰|：|“|汝有何|力|，|敢|小觑|我|！|”|方欲|交兵|，|刘表|便退|。|坚|纵马|赶去|，|两|山后|伏兵|齐|起|，|背后|蔡瑁|、|蒯越|赶来|，|将|孙坚|困在垓心|。|正是|：|玉玺|得|来|无用|处|，|反|因此|宝|动|刀|兵|。|毕竟|孙坚|怎|地|脱身|，|且听下文分解|。|



|第七|回| |袁绍|磐|河|战|公孙| |孙坚|跨|江|击|刘表|

    		|

    |却说|孙坚|被|刘表|围住|，|亏|得|程普|、|黄盖|、|韩当|三将|死|救|得脱|，|折兵大半|，|夺路|引兵|回江东|。|自此|孙坚|与|刘表|结|怨|。|

    |且说|袁绍|屯兵|河内|，|缺|少|粮草|。|冀州|牧|韩馥|，|遣人送|粮|以资|军|用|。|谋士|逢纪|说|绍曰|：|“|大丈夫|纵横天下|，|何|待人|送|粮|为食|！|冀州|乃|钱粮|广|盛|之地|，|将军何不|取之|？|”|绍曰|：|“|未有|良策|。|”|纪|曰|：|“|可|暗使人|驰|书与|公孙瓒|，|令|进兵|取|冀州|，|约|以|夹攻|，|瓒|必|兴兵|。|韩馥|无谋|之辈|，|必|请将军|领|州事|；|就|中取事|，|唾手可得|。|”|绍|大喜|，|即|发书|到|瓒|处|。|瓒|得书|，|见|说|共攻|冀州|，|平|分|其地|，|大喜|，|即日|兴兵|。|

    |绍|却|使人|密报|韩馥|。|馥|慌|聚|荀|谌|、|辛评|二|谋士|商议|。|谌曰|：|“|公孙瓒|将|燕|、|代|之众|，|长驱|而来|，|其锋|不可当|。|兼|有|刘备|、|关|、|张|助之|，|难以|抵敌|。|今|袁本初|智勇|过人|，|手下|名将|极广|，|将军可|请|彼|同|治|州事|，|彼必|厚待|将军|，|无患|公孙瓒|矣|。|”|韩馥|即差|别驾|关|纯|去|请|袁绍|。|长史|耿武|谏曰|：|“|袁绍|孤|客|穷|军|，|仰|我|鼻|息|，|譬|如|婴|儿|在|股|掌|之上|，|绝其|�|�|�|�|，|立|可|饿|死|。|奈何|欲以|州事|委|之|？|此|引|虎|入|羊|群|也|。|”|馥|曰|：|“|吾乃|袁氏|之故|吏|，|才|能|又|不如|本初|。|古|者|择|贤|者|而|让|之|，|诸君|何|嫉妒|耶|？|”|耿武|叹曰|：|“|冀州|休矣|！|”|于

In [46]:
decode(encode(text[:1000], pair_to_code), vocab) == text[:1000]

True